In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import numpy as np 
import pandas as pd

In [3]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
import scanpy as sc
import anndata as ad
import mudata as md

In [5]:
sys.path.append("/home/chenxufeng/WorkSpace/chenxufeng_MASLDHCC_2025/MASLDHCC_reproducibility/src")
import sc_helpers as sch

## Configurations

In [6]:
%matplotlib inline

sch.pl.validate_and_load_fonts(["Arial"])

matplotlib.rcParams["figure.figsize"] = [4, 4]
matplotlib.rcParams["figure.dpi"] = 100
matplotlib.rcParams["savefig.dpi"] = 300
matplotlib.rcParams["image.cmap"] = "Spectral_r"
matplotlib.rcParams["font.family"] = "Arial"
matplotlib.rcParams["grid.alpha"] = 0

sc.settings.verbosity = 0
sc.settings.set_figure_params(
    dpi=100, 
    facecolor="white",
    frameon=False,
)

In [7]:
import warnings
from numba.core.errors import NumbaDeprecationWarning
warnings.simplefilter("ignore", category=NumbaDeprecationWarning)
warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)

In [8]:
dirPjtHome = "/mnt/TrueNas/project/chenxufeng/Data/PMID33098772_Cell2020_Skin/"
os.chdir(dirPjtHome)

In [9]:
!lsd -lh 1_AnnData/

.rw-r-----. chenxufeng sysbio 1.2 GB Wed Jun  5 16:01:17 2024  atac_data.h5ad
.rw-r-----. chenxufeng sysbio 1.4 GB Sat Sep 21 15:50:44 2024  hair_follicle_shareseq_atac_preprocessed.h5ad
.rw-r-----. chenxufeng sysbio 546 MB Sat Sep 21 16:49:14 2024  hair_follicle_shareseq_rna_preprocessed.h5ad
.rw-r-----. chenxufeng sysbio 1.5 GB Thu Apr  9 18:31:12 2026  mouse_skin_hf_preprocessed.h5ad
.rw-r-----. chenxufeng sysbio 3.8 GB Thu Apr  9 18:30:04 2026  mouse_skin_ife_preprocessed.h5ad
.rw-r-----. chenxufeng sysbio 136 MB Wed Jun  5 16:01:15 2024  rna_data.h5ad
.rw-r-----. chenxufeng sysbio 7.9 MB Tue Apr 19 23:23:08 2022  shareseq.hair_follicle.joint_representation.h5ad
drwxr-xr-x. chenxufeng sysbio   4 B  Wed Apr 20 03:22:06 2022  shareseq_annotated_data


## Load the data

In [31]:
mdata = md.read("benchmark/data/mouse_skin-hf_benchmark.h5mu")

In [26]:
mdata = md.read("/mnt/TrueNas/project/chenxufeng/Data/PMID36973557_NatBiotechnol2023_T-cell-depleted/benchmark/data/t-cell-depleted-bm_benchmark.h5mu")

In [27]:
mdata

MuData object with n_obs × n_vars = 8627 × 233703
  2 modalities
    ATAC:	8627 x 216477
      obs:	'Sample', 'TSSEnrichment', 'ReadsInTSS', 'ReadsInPromoter', 'ReadsInBlacklist', 'PromoterRatio', 'PassQC', 'NucleosomeRatio', 'nMultiFrags', 'nMonoFrags', 'nFrags', 'nDiFrags', 'BlacklistRatio', 'Clusters', 'ReadsInPeaks', 'FRIP', 'leiden', 'phenograph', 'celltype', 'SEACell', 'sample'
      var:	'seqnames', 'start', 'end', 'width', 'strand', 'score', 'replicateScoreQuantile', 'groupScoreQuantile', 'Reproducibility', 'GroupReplicate', 'nearestGene', 'distToGeneStart', 'peakType', 'distToTSS', 'nearestTSS', 'GC', 'idx', 'N', 'Bcells_primed', 'Bcells_lineage_specific'
      uns:	'FIMOColumns', 'GeneScoresColumns', 'InSilicoChipColumns', 'celltype_colors', 'celltype_combined_colors', 'leiden', 'leiden_colors', 'neighbors', 'phenograph_colors', 'tab20', 'umap'
      obsm:	'DM_EigenVectors', 'GeneScores', 'X_svd', 'X_umap'
      varm:	'FIMO', 'InSilicoChip', 'InSilicoChip_Corrs', 'OpenPeaks'
      layers:	'counts', 'tf_idf'
      obsp:	'ImputeWeights', 'connectivities', 'distances'
    RNA:	8627 x 17226
      obs:	'sample', 'celltype', 'palantir_pseudotime', 'macrostates_fwd', 'clusters_gradients', 'term_states_fwd', 'term_states_fwd_probs', 'init_states_fwd', 'init_states_fwd_probs'
      var:	'highly_variable', 'means', 'dispersions', 'dispersions_norm'
      uns:	'T_fwd_params', 'celltype_colors', 'clusters_gradients_colors', 'coarse_fwd', 'custom_branch_mask_columns', 'dea_celltype', 'dendrogram_celltype', 'eigendecomposition_fwd', 'hvg', 'init_states_fwd_colors', 'lineage_colors', 'log1p', 'macrostates_fwd_colors', 'neighbors', 'pca', 'sample_colors', 'schur_matrix_fwd', 'term_states_fwd_colors', 'umap'
      obsm:	'T_fwd_umap', 'X_FDL', 'X_fate_simplex_fwd', 'X_pca', 'X_umap', 'branch_masks', 'cell_state_masks', 'cellrank_branch_masks', 'cellrank_fate_probabilities', 'cellrank_masks', 'init_states_fwd_memberships', 'lineages_fwd', 'macrostates_fwd_memberships', 'palantir_branch_probs', 'palantir_fate_probabilities', 'palantir_lineage_cells', 'schur_vectors_fwd'
      varm:	'PCs', 'geneXTF'
      layers:	'MAGIC_imputed_data', 'counts'
      obsp:	'connectivities', 'distances', 'knn'

In [32]:
adata = mdata["ATAC"]

In [28]:
np.min(adata.X), np.max(adata.X)

(0, 1)

In [30]:
np.min(adata.X), np.max(adata.X)

(0.0, 29.0)

## Preprocessing

In [18]:
adata

AnnData object with n_obs × n_vars = 6260 × 334124
    obs: 'topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4', 'topic_5', 'topic_6', 'topic_7', 'topic_8', 'topic_9', 'topic_10', 'topic_11', 'topic_12', 'topic_13', 'topic_14', 'topic_15', 'topic_16', 'topic_17', 'topic_18', 'topic_19', 'topic_20', 'topic_21', 'topic_22', 'topic_23', 'true_cell', 'softmax_denom'
    var: 'chr', 'start', 'end', 'endogenous'
    uns: 'TSS_metadata', 'distance_to_TSS_genes', 'motifs', 'neighbors', 'topic_dendogram', 'true_cell_colors', 'umap'
    obsm: 'X_joint_umap_features', 'X_topic_compositions', 'X_umap', 'X_umap_features'
    varm: 'distance_to_TSS', 'motifs_hits', 'topic_feature_activations', 'topic_feature_compositions'
    obsp: 'connectivities', 'distances'

In [19]:
import re
import numpy as np
import pandas as pd
import scanpy as sc

def lineage_vs_rest_regions_minimal(
    atac,                 # AnnData
    cell_selected,        # DataFrame: index=cell barcodes, 有 lineage 这一列
    lineage: str,
    fdr: float = 0.05,
    min_logfc: float = 0.0,
):
    # ---------- helpers (内联，避免外部依赖) ----------
    def to_bool_mask(series: pd.Series) -> pd.Series:
        """把常见 truthy 值统一转 bool。"""
        if series.dtype == bool:
            return series
        s = series.astype(str).str.strip().str.lower()
        truthy = {"1", "true", "t", "yes", "y"}
        return s.isin(truthy)

    def parse_region(region: str):
        """
        支持:
          - chr1:100-200
          - chr1-100-200
        """
        text = str(region).strip().replace(":", "-", 1)
        m = re.fullmatch(r"(.+)-(\d+)-(\d+)", text)
        if m is None:
            raise ValueError(f"Unsupported region format: {region!r}")
        chrom, start, end = m.groups()
        return chrom, int(start), int(end)

    def names_to_bed(names: pd.Index) -> pd.DataFrame:
        rows = [parse_region(x) for x in pd.Index(names).astype(str)]
        return (
            pd.DataFrame(rows, columns=["chr", "start", "end"])
            .sort_values(["chr", "start", "end"])
            .reset_index(drop=True)
        )

    def filter_markers(rank_df: pd.DataFrame, fdr: float, min_logfc: float) -> pd.DataFrame:
        if not (0 < fdr <= 1):
            raise ValueError(f"fdr must be in (0, 1], got {fdr}")
        if "names" not in rank_df.columns or "pvals_adj" not in rank_df.columns:
            raise ValueError("rank_genes_groups_df result must contain columns: names, pvals_adj")

        mask = rank_df["pvals_adj"].fillna(np.inf) <= fdr
        if "logfoldchanges" in rank_df.columns:
            mask &= rank_df["logfoldchanges"].fillna(-np.inf) > min_logfc
        if {"pct_nz_group", "pct_nz_reference"}.issubset(rank_df.columns):
            mask &= rank_df["pct_nz_group"].fillna(0) > rank_df["pct_nz_reference"].fillna(0)

        out = rank_df.loc[mask].copy()
        if out.empty:
            return out
        sort_cols = [c for c in ["pvals_adj", "scores"] if c in out.columns]
        if sort_cols:
            asc = [True if c == "pvals_adj" else False for c in sort_cols]
            out = out.sort_values(sort_cols, ascending=asc, na_position="last")
        return out.reset_index(drop=True)

    # ---------- 1) 对齐细胞 ----------
    shared = atac.obs_names.intersection(cell_selected.index)
    if len(shared) == 0:
        raise ValueError(f"[{lineage}] no overlap between atac.obs_names and cell_selected.index")

    if lineage not in cell_selected.columns:
        raise ValueError(f"[{lineage}] column not found in cell_selected")

    # ---------- 2) 构造 lineage vs rest ----------
    lineage_series = cell_selected.reindex(shared)[lineage].fillna(False)
    lineage_mask = to_bool_mask(lineage_series)

    n_lineage = int(lineage_mask.sum())
    n_rest = int((~lineage_mask).sum())
    if n_lineage == 0:
        raise ValueError(f"[{lineage}] no lineage cells after alignment")
    if n_rest == 0:
        raise ValueError(f"[{lineage}] no rest cells after alignment")

    sub = atac[shared].copy()
    sub.obs["__group__"] = pd.Categorical(
        np.where(lineage_mask.values, lineage, "rest"),
        categories=[lineage, "rest"],
    )

    # ---------- 3) 差异分析 ----------
    sc.tl.rank_genes_groups(
        sub,
        groupby="__group__",
        groups=[lineage],
        reference="rest",
        method="wilcoxon",
        use_raw=False,
        pts=True,
    )
    rank_df = sc.get.rank_genes_groups_df(sub, group=lineage)

    # ---------- 4) 过滤 ----------
    filtered_df = filter_markers(rank_df, fdr=fdr, min_logfc=min_logfc)
    if filtered_df.empty:
        raise ValueError(
            f"[{lineage}] no peaks passed filter (pvals_adj <= {fdr}, logfoldchanges > {min_logfc})"
        )

    # ---------- 5) 转 BED ----------
    bed_df = names_to_bed(pd.Index(filtered_df["names"].astype(str).unique()))

    meta = {
        "lineage": lineage,
        "n_shared_cells": int(len(shared)),
        "n_lineage_cells": n_lineage,
        "n_rest_cells": n_rest,
        "fdr_threshold": float(fdr),
        "min_logfc": float(min_logfc),
        "n_selected_regions": int(bed_df.shape[0]),
    }

    return bed_df, filtered_df, rank_df, meta

In [23]:
cell_selected = pd.read_csv("/mnt/TrueNas/project/chenxufeng/Data/PMID33098772_Cell2020_Skin/benchmark/cell_list/cell_state_masks_HF.csv", index_col=0)

In [24]:
cell_selected.head()

,IRS,Cortex,Medulla
barcode,,,
R1.01.R2.02.R3.21.P1.56,True,False,False
R1.01.R2.03.R3.61.P1.54,False,False,False
R1.01.R2.05.R3.79.P1.54,False,False,True
R1.01.R2.06.R3.64.P1.56,False,False,False
R1.01.R2.06.R3.83.P1.55,True,False,False


In [25]:
bed_df, filtered_df, rank_df, meta = lineage_vs_rest_regions_minimal(
    atac=adata,
    cell_selected=cell_selected,
    lineage="IRS",
    fdr=0.05,
    min_logfc=0.0,
)

meta
bed_df.head()
filtered_df.head()

ValueError: [IRS] no peaks passed filter (pvals_adj <= 0.05, logfoldchanges > 0.0)